In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
from analyses import io

In [ ]:
colors = cl.StackPalette.load("safe")

In [ ]:
celldf = io.read_celldfs(
    "../runs/invading/", 
    levels=["replica", "cell_energy", "mul_energy"]
).with_columns(
    pl.col("cell_energy").str.strip_prefix("cell_energy-").cast(pl.UInt32),
    pl.col("mul_energy").str.strip_prefix("mul_energy-").cast(pl.UInt32),
    pl.col("replica").cast(pl.UInt32),
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5,
).with_columns(
    mul_gamma=20 - pl.col("mul_energy"),
    cell_gamma=20 - pl.col("cell_energy"),
)
celldf

In [ ]:
grouppers = ["mul_gamma", "cell_gamma"]
clusterdf = celldf.filter(pl.col("wtime") >= 4e6, lineage="mul").group_by(grouppers).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    cluster_displ=pl.col("displ").mean()
).sort(grouppers)
clusterdf

In [ ]:
pvdf = clusterdf.pivot(on="mul_gamma", index="cell_gamma", values="cluster_displ")
pvdf

In [ ]:
go.Figure(go.Heatmap(
    z=pvdf.drop("cell_gamma").to_numpy(),
    x=pvdf.columns[1:],
    y=pvdf["cell_gamma"]
))

In [ ]:
filterdf = celldf.filter(
    pl.col("wtime") >= 4e6,
    lineage="mul",
    mul_energy=12,
)
px.violin(
    filterdf,
    x="cell_gamma",
    y="displ"
).update_traces(
    jitter=1,
    marker_line_width=1,
    marker_line_color="white",
    marker_opacity=0.5,
    marker_color=colors[0]
).add_traces(
    px.scatter(
        filterdf.group_by("cell_gamma").median(),
        x="cell_gamma",
        y="displ"
    ).update_traces(
        marker_color="gray"
    ).data
).update_layout(
    template="plotly_white",
    width=400,
    height=300,
    showlegend=False,
    xaxis_dtick=1,
    # yaxis_range=[0, max_chem],
    yaxis_title="distance to peak"
)